<a href="https://colab.research.google.com/github/AlisonTDoyle/Machine-Learning-Lego-Brick-Identification/blob/main/Model_1_VGG_1_Layer_Removal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [37]:
import numpy as np
import pandas as pd
import os

from google.colab import drive
from PIL import Image
from sklearn.model_selection import train_test_split

import keras
from keras import layers
from keras import ops

# Data read-in and preparation

In [23]:
# mount drive to access lego brick images
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
# read in files
SMALL_LEGO_FOLDER_PATH = '/content/drive/MyDrive/Colab Notebooks/ML - CA2 (Lego Brick)/SmallLego'
NUMBER_OF_BRICK_TYPES = 5

image_data = []

for brick_type in range(NUMBER_OF_BRICK_TYPES):
    subfolder_path = os.path.join(SMALL_LEGO_FOLDER_PATH, str(brick_type))

    # Check if the subfolder exists
    if os.path.exists(subfolder_path):
        # Loop through each image in the subfolder
        for image_name in os.listdir(subfolder_path):
            image_path = os.path.join(subfolder_path, image_name)

            # Open the image and convert it to a numpy array
            img = Image.open(image_path)
            img_array = np.array(img)

            # Append the image data to the list
            # image_data.append({
            #     'brick_type': brick_type,
            #     'image_name': image_name,
            #     'image_path': image_path,
            #     'image_data': img_array
            # })
            image_data.append(img_array)

# Create a DataFrame from the list
# df = pd.DataFrame(image_data)

# Display the DataFrame
# print(df.head())

In [ ]:
# resize images to meet imagenet standards (224 x 224 x 3)

In [33]:
FIRST_SPLIT_RATIO = 0.4
SECOND_SPLIT_RATIO = 0.5
SEED = 505

# split data so that training receives 60% of all images
training_data, other = train_test_split(
    image_data,
    test_size = FIRST_SPLIT_RATIO,
    random_state = SEED,
    shuffle = True
)

# split the remaining 40% amongst validation and testing
validation_data, testing_data = train_test_split(
    other,
    test_size=SECOND_SPLIT_RATIO,
    random_state=SEED,
    shuffle=True
)

# Model Training and Testing

In [42]:
# initialise base model
basic_vgg16_model = keras.applications.VGG16(
    weights="imagenet",
    name="vgg16"
)

In [43]:
# creating summary to make sure model was created alright
basic_vgg16_model.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# create new model


lego_model = object()

In [40]:
EPOCHS = 10
# train new model on lego bricks
lego_model.compile()
lego_model.fit(training_data, epochs=EPOCHS)

lego_model.summary(show_trainable=True)

# Resources used
- [Keras - Transfer Learning and Fine-Tuning](https://keras.io/guides/transfer_learning/)
- [Keras - VGG16 and VGG19 Models](https://keras.io/api/applications/vgg/vgg_models/#vgg16-function)
